# Metagenomic Analyses of the Longitudinal Acne Study
## *Cutibacterium acnes* strain level analyses

Date last updated: 1/1/2026

Notebook author: Yang Chen

This notebook plots the following:

- Relative abundance plots for *Cutibacterium acnes* strain types

In [30]:
# Import Python packages
import pandas as pd
import biom
import matplotlib.pyplot as plt
import seaborn as sns
import os
from itertools import cycle

In [31]:
# Switch this to Genus to make the Genus level plots and Species to make the  plots
# taxa_level = 'Genus'
taxa_level = 'Species'

In [32]:
# Read in metadata
md = pd.read_csv('../Metadata/metadata_final_22102024.tsv', sep='\t')
md['zone'] = md['zone'].replace({'Lesional': 'L', 'Non-lesional': 'NL'})
md['label'] = md['subject_ID'].astype(str) + '_' + 'sev-'+ md['local_lesion_severity'].astype(str) + '_' + md['zone']
md = md.sort_values(by=['subject_ID', 'local_lesion_severity'])
md.index = md['SampleID']
md.head()

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group,severity_level,severity_group,label
SampleID,,,,,,,,,,,,,,,,,,,,,
LAMI.RD001.D14.C1,LAMI.RD001.D14.C1,C1,not applicable,NL,skin,Day 14,not applicable,14,1,not applicable,...,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_sev-0_NL
LAMI.RD001.D28.C1,LAMI.RD001.D28.C1,C1,not applicable,NL,skin,Day 28,not applicable,28,1,not applicable,...,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_sev-0_NL
LAMI.RD001.D0.C2,LAMI.RD001.D0.C2,C2,not applicable,NL,skin,Day 0,not applicable,0,1,not applicable,...,control,RD001,healthy,PP_1,PP_1C2,Non-lesional_C2,Healthy,absent,absent Healthy,PP_1_sev-0_NL
LAMI.RD001.D28.C2,LAMI.RD001.D28.C2,C2,not applicable,NL,skin,Day 28,not applicable,28,1,not applicable,...,control,RD001,healthy,PP_1,PP_1C2,Non-lesional_C2,Healthy,absent,absent Healthy,PP_1_sev-0_NL
LAMI.RD001.D0.C1,LAMI.RD001.D0.C1,C1,not applicable,NL,skin,Day 0,not applicable,0,1,not applicable,...,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_sev-0_NL


In [33]:
# Define paths to the collapsed taxa tables
biom_paths = {
    'veba_MAGs_species': '../Data/metaG/Tables/metaG_mags_species.biom'}

In [34]:
# Define a function to modify the column names
def modify_column_name(col):
    return '_'.join(col.split('_')[:4])

In [35]:
# Read in input file
df = pd.read_csv('../Data/metaG/Tables/X_mags.with_taxonomy.with_slcs_c.acnes.csv')

# Assuming 'name_group' is the column containing the full names
# Extract the part that starts with "Cutibacterium acnes Type" and assign it to a new column 'name_group'
df['name.group'] = df['name'].str.extract(r'(Cutibacterium acnes Type \S+)')
df

,SLC,sample_mag,organism_type,taxonomy,name,LAMI_RD308_D9_C3_S18_L005,LAMI_RD306_D28_C3_S12_L005,LAMI_RD310_D16_C2_S22_L005,LAMI_RD304_D14_C3_S4_L005,LAMI_RD304_D11_C1_S3_L005,...,LAMI_RD308_D0_C2_S13_L005,LAMI_RD310_D0_C2_S19_L005,LAMI_RD306_D11_C1_S8_L005,LAMI_RD310_D0_C3_S20_L005,LAMI_RD310_D14_C3_S21_L005,LAMI_RD306_D14_C1_S9_L005,LAMI_RD310_D21_C2_S23_L005,LAMI_RD308_D14_C3_S16_L005,LAMI_RD304_D0_C1_S1_L005,name.group
0,PSLC-1,LAMI_RD306_D0_C3_S7_L005__MAXBIN2-40__P.1__bin...,prokaryotic,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,s__Cutibacterium acnes Type IA (MAG 1),777667,1513946,19467,127037,5905,...,1811,196880,825942,372845,441798,83571,196812,782381,111499,Cutibacterium acnes Type IA
1,PSLC-2,LAMI_RD306_D0_C3_S7_L005__METABAT2__P.1__bin.2,prokaryotic,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,NaN,14720,171159,1374,21838,896,...,27,20516,65240,26311,13760,5157,28370,7988,23177,NaN
2,PSLC-7,LAMI_RD306_D11_C1_S8_L005__CONCOCT__P.1__14_sub,prokaryotic,d__Bacteria;p__Pseudomonadota;c__Gammaproteoba...,NaN,5,74,0,35,1,...,0,1,102859,5,1,22,13,1,35,NaN
3,PSLC-11,LAMI_RD306_D11_C1_S8_L005__CONCOCT__P.1__26,prokaryotic,d__Bacteria;p__Pseudomonadota;c__Gammaproteoba...,NaN,31,69,0,3,0,...,0,1,764138,0,0,25,0,3,1,NaN
4,PSLC-3,LAMI_RD306_D11_C1_S8_L005__CONCOCT__P.1__27,prokaryotic,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,NaN,1217,72530,123,7890,258,...,4,661,94133,819,446,11267,970,635,2289,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,PSLC-1,LAMI_RD310_D16_C2_S22_L005__METABAT2__P.1__bin.1,prokaryotic,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,s__Cutibacterium acnes Type II (MAG 20),325344,98077,34139,416073,20568,...,694,551756,70551,674986,371153,4985,705039,212634,404418,Cutibacterium acnes Type II
73,PSLC-1,LAMI_RD310_D21_C2_S23_L005__CONCOCT__P.1__37,prokaryotic,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,s__Cutibacterium acnes Type II (MAG 21),389589,171086,49066,459679,22935,...,832,848529,112684,879014,549818,8742,1076846,268975,448362,Cutibacterium acnes Type II
74,VSLC-6,LAMI_RD310_D7_C3_S24_L005__GENOMAD__Virus.1,viral,Viruses;Monodnaviria;Shotokuvirae;Cossaviricot...,NaN,0,38,0,0,0,...,0,4,10,6305,0,0,0,0,0,NaN
75,PSLC-1,LAMI_RD310_D7_C3_S24_L005__METABAT2__P.1__bin.1,prokaryotic,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,s__Cutibacterium acnes Type II (MAG 22),328321,91234,35262,438165,21590,...,659,574052,67200,719147,381184,4591,726124,209969,424201,Cutibacterium acnes Type II


In [36]:
# Subset to only rows with no NaN (not Cutibacterium acnes, so no strain info in name column)
df = df[df['name.group'].notna()]

# Subset DataFrame to rows where 'name_group' column starts with 'Cutibacterium acnes'
df = df[df['name.group'].str.startswith('Cutibacterium acnes')]

df['name.group'].value_counts()
# df['name.group'].value_counts()

name.group
Cutibacterium acnes Type II    11
Cutibacterium acnes Type IA     6
Cutibacterium acnes Type IB     5
Name: count, dtype: int64

### Number of each type of MAG

In [37]:
# Data and corresponding colors
data = {
    'Type IA': 6,
    'Type IB': 5,
    'Type II': 11
}

colors = {
    'Type IA': '#FFD1A9',  # Light orange
    'Type IB': '#FFA500',  # Mid orange
    'Type II': '#CC5500'   # Dark orange
}

name_group_series = pd.Series(data, name='count')

# Define the explicit order
ordered_index = ['Type IA', 'Type IB', 'Type II']
ordered_data = name_group_series[ordered_index]
ordered_colors = [colors[key] for key in ordered_index]

# Plot Horizontal Bar Chart
plt.figure(figsize=(5, 3))
ordered_data.plot(
    kind='barh', 
    color=ordered_colors, 
    alpha=0.8
)

# Add labels and title
plt.xlabel('Count', fontsize=12)
plt.title('MLST-typed Cutibacterium acnes strains', fontsize=14)

# Rotate y-axis labels
plt.yticks(rotation=45)

# Add data labels
for index, value in enumerate(ordered_data):
    plt.text(value, index, f'{value}', va='center')

# Show the plot
plt.tight_layout()
plt.show()

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_66155/1349648883.py:42: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [38]:
# # Remove first four columns
df = df.iloc[:, 4:]

# Set new index
df.set_index(df.columns[0], inplace=True)

# Convert all values to numeric, setting non-numeric values to NaN
df = df.apply(pd.to_numeric, errors='coerce')

# Apply the function to the column names
df.columns = [modify_column_name(col) for col in df.columns]

# Change sample names so they match metadata
df.columns = df.columns.str.replace('_', '.', regex=False)

df.head()

,LAMI.RD308.D9.C3,LAMI.RD306.D28.C3,LAMI.RD310.D16.C2,LAMI.RD304.D14.C3,LAMI.RD304.D11.C1,LAMI.RD306.D14.C3,LAMI.RD308.D0.C3,LAMI.RD306.D23.C1,LAMI.RD308.D23.C2,LAMI.RD310.D7.C3,...,LAMI.RD308.D0.C2,LAMI.RD310.D0.C2,LAMI.RD306.D11.C1,LAMI.RD310.D0.C3,LAMI.RD310.D14.C3,LAMI.RD306.D14.C1,LAMI.RD310.D21.C2,LAMI.RD308.D14.C3,LAMI.RD304.D0.C1,name.group
name,,,,,,,,,,,,,,,,,,,,,
s__Cutibacterium acnes Type IA (MAG 1),777667,1513946,19467,127037,5905,1529048,63022,1186104,450121,19208,...,1811,196880,825942,372845,441798,83571,196812,782381,111499,NaN
s__Cutibacterium acnes Type IA (MAG 2),775594,1522344,19719,127811,5994,1533464,63354,1197434,448513,19132,...,1809,198453,832734,374919,447003,84130,197854,783524,111435,NaN
s__Cutibacterium acnes Type IA (MAG 3),788086,1544556,20344,131326,6190,1559337,63969,1210542,457334,19473,...,1850,201364,844942,378610,457067,85865,201885,793252,114153,NaN
s__Cutibacterium acnes Type IA (MAG 4),780834,1535993,19895,128011,5951,1551302,63556,1199449,452006,19203,...,1869,200068,839140,377336,451537,85088,197051,787833,111607,NaN
s__Cutibacterium acnes Type IA (MAG 5),901353,981993,15592,120927,6642,1020013,71516,1343335,516896,14697,...,2453,152467,557413,273568,301208,55632,162549,870057,120580,NaN


In [39]:
# Predefined color palette for specific families
taxa_colors = {
    ' s__Cutibacterium acnes': '#ffa505',  # Bright orange
    ' s__Malassezia spp.': '#ec8cbf',     # Light purple pink
    ' s__Lawsonella cleavelandensis': '#70a8dc',      # Light blue
    ' s__Corynebacterium spp.': '#00008B',      # Dark blue
    ' s__Streptococcus spp.': '#e2b46c',    # Beige
    ' s__Micrococcus luteus': '#ffe59a',        # Pastel yellow
    ' s__Neisseria cerebrosa': '#f6475f',         # Redish pink
    ' s__Porphyromonas pasteri': '#c5bce0',         # Pastel purplish
    ' s__Actinomyces spp.': '#f4cccd',  # Light pink
    ' s__Alloprevotella spp.': '#bcbcbc',  # Light gray
    ' s__Psychrobacter spp.': '#daead3',  # Light greenish gray
    ' s__Marinomonas spp.' : '#9FE2BF', # Seafoam green
    ' s__Streptococcus mitis' : '#92f0f0',      # Fluorescent light blue
    'Others': '#ededed'                 # White
}

In [40]:
# Function to load BIOM table, collapse by taxa, and sort rows by row sum
def load_biom_table(biom_path):
    table = biom.load_table(biom_path)
    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before returning
    df = df.drop(columns=['row_sum'])
    
    return df

In [41]:
# Function to determine the top 15 in taxa level and collapse the rest as "Others"
def collapse_top_10(df):
    top_families = df.sum(axis=1).nlargest(10).index  # Select top 15 families
    df_top = df.loc[top_families]
    df_top.loc['Others'] = df.loc[~df.index.isin(top_families)].sum()
    return df_top

In [42]:
# Function to get or assign colors to families
def get_taxa_colors(families, global_taxa_color_map):
    for taxa in families:
        if taxa not in global_taxa_color_map:
            if taxa in taxa_colors:
                global_taxa_color_map[taxa] = taxa_colors[taxa]
    return global_taxa_color_map

In [43]:
def plot_relative_abundance(df, metadata, group_column, output_dir, biom_key, taxa_color_map):
    # Average by group
    df_grouped = df.groupby(metadata[group_column], axis=1).mean()
    
    # Reorder the columns to Acne Non-lesional then Lesional
    desired_order = ['Acne_NL', 'Acne_L']
    df_grouped = df_grouped[desired_order]

    # Create output file paths
    output_png_file = os.path.join(output_dir, f'{biom_key}_{taxa_level}_relative_abundance_plot.png')

    # Set plot title based on biom_key
    if biom_key == 'veba_MAGs_species':
        plot_title = f'metaG Relative Abundance'

    # Plot
    ax = df_grouped.T.plot(kind='bar', stacked=True, figsize=(10, 10),
                           width=0.8,  # Bars closer together
                           color=[taxa_color_map.get(fam, '#ADD8E6') for fam in df_grouped.index])

    plt.ylabel('Relative Abundance', fontsize=18)
    plt.xlabel(' ')
    plt.title(plot_title, fontsize=16)

    new_labels = ['Acne\nNon-lesional\n(n=12)', 'Acne\nLesional\n(n=12)']

    # Set the new x-tick labels
    plt.xticks(ticks=range(len(new_labels)), labels=new_labels, rotation=0, ha='center', fontsize=16)

    plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=16)
    plt.tight_layout()

    plt.savefig(output_png_file, format='png', dpi=600)
    plt.show()

    plt.close()


In [44]:
df_all_species = load_biom_table('../Data/metaG/Tables/metaG_mags_species.biom')
df_all_species = collapse_top_10(df_all_species)
df_all_species = df_all_species.transpose()

In [45]:
# Map 'group' values from metadata to sample names in the transposed DataFrame
df_all_species['group'] = df_all_species.index.map(
    lambda SampleID: md.loc[md['SampleID'] == SampleID, 'group'].values[0] if SampleID in md['SampleID'].values else None
)

# Group by the 'group' column and sum the values for each group
relative_abundance_grouped_nonles_les = df_all_species.groupby('group').sum()

relative_abundance_grouped_nonles_les.head()

,s__Cutibacterium acnes,s__Malassezia spp.,s__Lawsonella cleavelandensis,s__Corynebacterium spp.,s__Cutibacterium granulosum,s__Actinomyces spp.,s__Alloprevotella spp.,s__Micrococcus luteus,s__Neisseria cerebrosa,s__Streptococcus mitis,Others
group,,,,,,,,,,,
Acne_L,10.441045,0.779511,0.130420,0.132022,0.074342,0.050138,0.141438,0.045752,0.024969,0.029652,0.150710
Acne_NL,10.187829,0.904927,0.276288,0.208139,0.139458,0.097705,0.005818,0.069662,0.033738,0.022902,0.053534


In [46]:
# Normalize each row in the DataFrame so that each row sums to 1
relative_abundance_normalized = relative_abundance_grouped_nonles_les.div(relative_abundance_grouped_nonles_les.sum(axis=1), axis=0)

# Display the first few rows of the normalized DataFrame
relative_abundance_normalized.head()

,s__Cutibacterium acnes,s__Malassezia spp.,s__Lawsonella cleavelandensis,s__Corynebacterium spp.,s__Cutibacterium granulosum,s__Actinomyces spp.,s__Alloprevotella spp.,s__Micrococcus luteus,s__Neisseria cerebrosa,s__Streptococcus mitis,Others
group,,,,,,,,,,,
Acne_L,0.870087,0.064959,0.010868,0.011002,0.006195,0.004178,0.011786,0.003813,0.002081,0.002471,0.012559
Acne_NL,0.848986,0.075411,0.023024,0.017345,0.011622,0.008142,0.000485,0.005805,0.002811,0.001909,0.004461


In [47]:
# Remove the column 'name.group' from the DataFrame 'df' if it exists
if 'name.group' in df.columns:
    df = df.drop(columns=['name.group'])

df.head()    

,LAMI.RD308.D9.C3,LAMI.RD306.D28.C3,LAMI.RD310.D16.C2,LAMI.RD304.D14.C3,LAMI.RD304.D11.C1,LAMI.RD306.D14.C3,LAMI.RD308.D0.C3,LAMI.RD306.D23.C1,LAMI.RD308.D23.C2,LAMI.RD310.D7.C3,...,LAMI.RD304.D28.C3,LAMI.RD308.D0.C2,LAMI.RD310.D0.C2,LAMI.RD306.D11.C1,LAMI.RD310.D0.C3,LAMI.RD310.D14.C3,LAMI.RD306.D14.C1,LAMI.RD310.D21.C2,LAMI.RD308.D14.C3,LAMI.RD304.D0.C1
name,,,,,,,,,,,,,,,,,,,,,
s__Cutibacterium acnes Type IA (MAG 1),777667,1513946,19467,127037,5905,1529048,63022,1186104,450121,19208,...,362236,1811,196880,825942,372845,441798,83571,196812,782381,111499
s__Cutibacterium acnes Type IA (MAG 2),775594,1522344,19719,127811,5994,1533464,63354,1197434,448513,19132,...,362942,1809,198453,832734,374919,447003,84130,197854,783524,111435
s__Cutibacterium acnes Type IA (MAG 3),788086,1544556,20344,131326,6190,1559337,63969,1210542,457334,19473,...,374395,1850,201364,844942,378610,457067,85865,201885,793252,114153
s__Cutibacterium acnes Type IA (MAG 4),780834,1535993,19895,128011,5951,1551302,63556,1199449,452006,19203,...,363097,1869,200068,839140,377336,451537,85088,197051,787833,111607
s__Cutibacterium acnes Type IA (MAG 5),901353,981993,15592,120927,6642,1020013,71516,1343335,516896,14697,...,421960,2453,152467,557413,273568,301208,55632,162549,870057,120580


In [48]:
# Define the function to categorize each row based on the criteria
def categorize_cutibacterium(row_name):
    if row_name.startswith(' s__Cutibacterium acnes Type IA'):
        return ' s__Cutibacterium acnes Type IA'
    elif row_name.startswith(' s__Cutibacterium acnes Type IB'):
        return ' s__Cutibacterium acnes Type IB'
    elif row_name.startswith(' s__Cutibacterium acnes Type II'):
        return ' s__Cutibacterium acnes Type II'
    else:
        return 'Other'

# Apply the function to categorize each row in the DataFrame 'df'
df['Category'] = df.index.map(categorize_cutibacterium)

# Group the DataFrame by the new 'Category' column and sum the values
grouped_df = df.groupby('Category').sum()

# Transpose 
grouped_df = grouped_df.transpose()

In [49]:
# Scale each row in relative_abundance_grouped_df by the corresponding value from df_all_species[' s__Cutibacterium acnes']
cutibacterium_values = df_all_species[' s__Cutibacterium acnes']

# Scale the rows in relative_abundance_grouped_df so they sum to the values in cutibacterium_values
relative_abundance_grouped_df_scaled = grouped_df.div(grouped_df.sum(axis=1), axis=0).mul(cutibacterium_values, axis=0)

# Add a new column called 'sum' that contains the sum of each row in relative_abundance_grouped_df_scaled
relative_abundance_grouped_df_scaled['sum'] = relative_abundance_grouped_df_scaled.sum(axis=1)


In [50]:
# Remove the 'group' column from df_all_species if it exists
if 'group' in df_all_species.columns:
    df_all_species = df_all_species.drop(columns=['group'])

In [51]:
# Add the columns from relative_abundance_grouped_df_scaled to df_all_species by matching the sample IDs (index)
df_all_species_updated = df_all_species.join(relative_abundance_grouped_df_scaled, how='left')

In [52]:
# Remove the 'sum' column from df_all_species if it exists
if 'sum' in df_all_species_updated.columns:
    df_all_species_updated = df_all_species_updated.drop(columns=['sum'])

In [53]:
# Move the last 3 columns to be the first 3 columns in df_all_species_updated
columns_to_move = df_all_species_updated.columns[-3:]  # Select the last 3 columns
other_columns = df_all_species_updated.columns[:-3]     # Select all other columns

# Rearrange the columns
df_all_species_reordered = df_all_species_updated[columns_to_move.tolist() + other_columns.tolist()]

# Display the updated DataFrame
df_all_species_reordered.head()


,s__Cutibacterium acnes Type IA,s__Cutibacterium acnes Type IB,s__Cutibacterium acnes Type II,s__Cutibacterium acnes,s__Malassezia spp.,s__Lawsonella cleavelandensis,s__Corynebacterium spp.,s__Cutibacterium granulosum,s__Actinomyces spp.,s__Alloprevotella spp.,s__Micrococcus luteus,s__Neisseria cerebrosa,s__Streptococcus mitis,Others
LAMI.RD308.D9.C3,0.310831,0.396100,0.251290,0.958221,0.006704,0.001253,0.004631,0.017226,0.004737,0.000291,0.004642,0.000837,0.000937,0.000521
LAMI.RD306.D28.C3,0.436272,0.189311,0.061186,0.686769,0.135709,0.038326,0.044729,0.022959,0.017136,0.000169,0.020260,0.004980,0.000769,0.028195
LAMI.RD310.D16.C2,0.188354,0.135180,0.666968,0.990502,0.005728,0.000167,0.000651,0.002413,0.000181,0.000002,0.000094,0.000080,0.000010,0.000172
LAMI.RD304.D14.C3,0.082335,0.090374,0.529078,0.701787,0.127238,0.035929,0.038021,0.022489,0.028849,0.004151,0.010036,0.009000,0.019085,0.003416
LAMI.RD304.D11.C1,0.110206,0.146420,0.732339,0.988965,0.001060,0.000133,0.000202,0.009299,0.000113,0.000031,0.000093,0.000052,0.000025,0.000026


In [54]:
# Remove the ' s__Cutibacterium acnes' column from df_all_species_reordered if it exists
if ' s__Cutibacterium acnes' in df_all_species_reordered.columns:
    df_all_species_reordered = df_all_species_reordered.drop(columns=[' s__Cutibacterium acnes'])

In [55]:
# Map 'group' values from metadata to sample names in the transposed DataFrame
df_all_species_reordered['group'] = df_all_species.index.map(
    lambda SampleID: md.loc[md['SampleID'] == SampleID, 'group'].values[0] if SampleID in md['SampleID'].values else None
)

# Group by the 'group' column and sum the values for each group
df_all_species_reordered_nonles_les = df_all_species_reordered.groupby('group').sum()

df_all_species_reordered_nonles_les.head()

,s__Cutibacterium acnes Type IA,s__Cutibacterium acnes Type IB,s__Cutibacterium acnes Type II,s__Malassezia spp.,s__Lawsonella cleavelandensis,s__Corynebacterium spp.,s__Cutibacterium granulosum,s__Actinomyces spp.,s__Alloprevotella spp.,s__Micrococcus luteus,s__Neisseria cerebrosa,s__Streptococcus mitis,Others
group,,,,,,,,,,,,,
Acne_L,3.090915,2.694293,4.655838,0.779511,0.130420,0.132022,0.074342,0.050138,0.141438,0.045752,0.024969,0.029652,0.150710
Acne_NL,3.246781,2.518748,4.422300,0.904927,0.276288,0.208139,0.139458,0.097705,0.005818,0.069662,0.033738,0.022902,0.053534


In [56]:
# Normalize each row in df_all_species_reordered_nonles_les so that the sum of each row equals 1
df_all_species_reordered_nonles_les_normalized = df_all_species_reordered_nonles_les.div(df_all_species_reordered_nonles_les.sum(axis=1), axis=0)

# Display the normalized DataFrame
df_all_species_reordered_nonles_les_normalized.head()


,s__Cutibacterium acnes Type IA,s__Cutibacterium acnes Type IB,s__Cutibacterium acnes Type II,s__Malassezia spp.,s__Lawsonella cleavelandensis,s__Corynebacterium spp.,s__Cutibacterium granulosum,s__Actinomyces spp.,s__Alloprevotella spp.,s__Micrococcus luteus,s__Neisseria cerebrosa,s__Streptococcus mitis,Others
group,,,,,,,,,,,,,
Acne_L,0.257576,0.224524,0.387986,0.064959,0.010868,0.011002,0.006195,0.004178,0.011786,0.003813,0.002081,0.002471,0.012559
Acne_NL,0.270565,0.209896,0.368525,0.075411,0.023024,0.017345,0.011622,0.008142,0.000485,0.005805,0.002811,0.001909,0.004461


In [57]:
# Define colors
general_colors = {
    ' s__Cutibacterium acnes Type IA': '#FFD1A9',  # Light orange
    ' s__Cutibacterium acnes Type IB': '#FFA500',  # Mid orange
    ' s__Cutibacterium acnes Type II': '#CC5500',  # Dark orange
    ' s__Malassezia spp.': '#ec8cbf',     # Light purple pink
    ' s__Lawsonella cleavelandensis': '#70a8dc',      # Light blue
    ' s__Corynebacterium spp.': '#00008B',      # Dark blue
    ' s__Cutibacterium granulosum': '#7B3F00',    # Rust
    ' s__Micrococcus luteus': '#ffe59a',        # Pastel yellow
    ' s__Neisseria cerebrosa': '#f6475f',         # Redish pink
    ' s__Porphyromonas pasteri': '#c5bce0',         # Pastel purplish
    ' s__Actinomyces spp.': '#f4cccd',  # Light pink
    ' s__Alloprevotella spp.': '#8c5079',  # Plum
    ' s__Psychrobacter spp.': '#daead3',  # Light greenish gray
    ' s__Marinomonas spp.' : '#9FE2BF', # Seafoam green
    ' s__Streptococcus mitis' : '#92f0f0',      # Fluorescent light blue
    'Others': '#ededed'                 # White
}

In [58]:
# Create a list of colors by mapping each column name in df_all_species_reordered_nonles_les_normalized to a color
color_list = [general_colors.get(col, '#ededed') for col in df_all_species_reordered_nonles_les_normalized.columns]

# Plot the ordered DataFrame as a stacked bar plot
new_labels = ['Acne\nNon-lesional\n(n=12)', 'Acne\nLesional\n(n=12)']
ax = df_all_species_reordered_nonles_les_normalized.plot(kind='bar', width=0.8, stacked=True, figsize=(10, 10), color=color_list)

# Set labels and title
plt.ylabel('Relative Abundance', fontsize=20)
plt.tick_params(axis='y', labelsize=16)  # Adjust labelsize as needed

plt.xticks(ticks=range(len(new_labels)), labels=new_labels, rotation=0, ha='center', fontsize=20)
plt.xlabel('')
plt.title('metaG Relative Abundance', fontsize=24)

# Adjust legend
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=16)
plt.tight_layout()

# Save the figure
plt.savefig('../Figures/Supplementary/Suppl_Figure_5/metaG_species_and_strains_relative_abundance.png', dpi=600)

# Show the plot
plt.show()

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_66155/2608588004.py:24: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
